# Orchestrator Eksperimen: Analisis Sentimen & Emosi

Notebook ini bersih dari *logic* panjang. Semua proses *loading* data, pelatihan model, penebangan (*logging*), dan komputasi metrik dipusatkan pada package `src/`.

## 1. Import Modul Terpusat

In [ ]:
import torch
from src import (
    setup_run_env, set_seed,
    build_full_pipeline,
    build_model_v as build_model, get_default_config,
    get_optimizer, get_scheduler,
    fit, compute_metrics,
    plot_training_curves, plot_confusion_matrix
)
from src.logger import get_logger

# Set Global Seed untuk reproduktifitas
set_seed(42)

## 2. Inisiasi Lingkungan *Run* (Logger & Direktori)

In [ ]:
EXPERIMENT_NAME = "baseline_bilstm"
run_env = setup_run_env(EXPERIMENT_NAME, base_out_dir="../outputs")

# Mengambil instance logger untuk orchestrator ini
logger = get_logger("orchestrator")
logger.info("Orchestrator siap dijalankan!")

## 3. Persiapan Data (Pipeline)

In [ ]:
logger.info("Membangun pipeline data...")
pipeline_out = build_full_pipeline(
    csv_path="../data/clean/cleaned_dataset.csv",
    text_column="clean_review",
    batch_size=64,
    max_seq_len=64,
    max_vocab_size=10000,
    random_seed=42
)

train_loader = pipeline_out["loaders"]["train"]
val_loader = pipeline_out["loaders"]["val"]
test_loader = pipeline_out["loaders"]["test"]
vocab = pipeline_out["vocab"]

## 4. Persiapan Model & Optimizer

In [ ]:
# Konfigurasi Model (Bisa ganti ke 'improved' atau 'large')
config = get_default_config("baseline")
config.update({
    "vocab_size": len(vocab),
    "num_epochs": 10,  # Ubah sesuai kebutuhan
    "use_amp": torch.cuda.is_available()  # Automatic Mixed Precision pakai GPU
})

logger.info(f"Membangun Model: {config['model_name']}")
model = build_model(config)
optimizer = get_optimizer(model, config)
scheduler = get_scheduler(optimizer, config)

## 5. Training Pipeline

In [ ]:
logger.info("Memulai Training Loop...")

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    config=config,
    run_env=run_env # Pass run environment (logs, checkpoints dirs)
)

logger.info("Training selesai!")

## 6. Visualisasi & Evaluasi Akhir

In [ ]:
logger.info("Melakukan plot kurva pembelajaran...")
plot_training_curves(
    history, 
    save_dir=run_env["fig_dir"], 
    filename="training_curves.png" 
)

logger.info("Evaluasi pada Test Set...")
# ... (Load best config lalu predict pada dataloader_test, diikuti dengan plotting confusion matrix menggunakan plot_confusion_matrix())
logger.info("Seluruh pipeline eksperimen terjalankan dengan baik!")